# Train Plain-English Medicine LoRA (Colab)

Runtime: **T4 GPU** (Runtime → Change runtime type → T4 GPU).

Before running, open Colab Secrets (key icon in the left panel) and add:

| Name | Value |
|------|-------|
| `HF_TOKEN` | your Hugging Face write token |

## 1. Install dependencies

In [ ]:
!pip install -q torch transformers peft accelerate bitsandbytes datasets trl sentencepiece huggingface-hub

## 2. Authenticate with Hugging Face

Pulls the token from Colab Secrets so it is never committed to the notebook.

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))

## 3. Clone the repo

In [ ]:
!git clone https://github.com/ksolano220/plain-english-medicine.git
%cd plain-english-medicine

## 4. Fetch + prepare the datasets

In [ ]:
!python src/fetch_data.py
!python src/prepare_dataset.py

## 5. Train the LoRA

About 3 to 5 hours on a T4. The trainer pushes to `ksolano220/plain-english-medicine` on the Hub at the end.

In [ ]:
import sys
sys.path.insert(0, 'src')

from train import train

trainer = train(
    train_path='data/processed/train.jsonl',
    val_path='data/processed/val.jsonl',
    hub_repo_id='ksolano220/plain-english-medicine',
)

## 6. Smoke test the trained adapter

In [ ]:
from inference import load, generate

model, tokenizer = load('ksolano220/plain-english-medicine')

test = 'Patient presents with acute exacerbation of chronic obstructive pulmonary disease requiring supplemental oxygen.'
print(generate(model, tokenizer, test))

## 7. Run evaluation

In [ ]:
!pip install -q sacrebleu rouge-score textstat
!python src/evaluate.py